# Basic Netrun Example

This notebook demonstrates:
1. Loading a network configuration from JSON
2. Creating and starting a Net
3. Injecting data into the network
4. Running the network until all processing is complete
5. Retrieving results from output queues

## Load the Network Configuration

In [ ]:
import json
from pathlib import Path

from netrun.core import Net, NetConfig

# Load the network configuration from JSON
config_path = Path("main.netrun.json")
config_data = json.loads(config_path.read_text())
config = NetConfig.model_validate(config_data)

print("Loaded config with nodes:")
for node in config.graph.nodes:
    print(f"  - {node.name}")

## Create and Run the Network

The network flow is:
```
double(x=5) → 10 → add(a=10, b=10) → 20 → format_result(value=20) → "The answer is: 20"
```

In [ ]:
async def run_network():
    async with Net(config) as net:
        # Inject input data:
        # - 'double' node receives x=5
        # - 'add' node receives b=10
        net.inject_data("double", "x", [5])
        net.inject_data("add", "b", [10])

        # Run until all processing is complete
        while True:
            # Move packets through edges
            await net.run_until_blocked()

            # Execute any startable epochs
            startable = net.get_startable_epochs()
            if not startable:
                break

            for epoch_id in startable:
                await net.execute_epoch(epoch_id)

        # Retrieve results from the output queue
        results = net.get_all_outputs("results")

        print("=" * 50)
        print("Results:")
        for packet in results:
            print(f"  {packet.value}")

        # Show captured print logs from all nodes
        print()
        print("Node Logs:")
        for node_name in ["double", "add", "format_result"]:
            logs = net.get_node_log(node_name)
            if logs:
                print(f"\n  [{node_name}]")
                for timestamp, message in logs:
                    print(f"    {timestamp.strftime('%H:%M:%S.%f')[:-3]} | {message}", end="")

# Run the network
await run_network()

## Understanding the Node Functions

The node functions are defined in `nodes.py`. Let's look at them:

In [ ]:
print(Path("nodes.py").read_text())

## Understanding the Network Configuration

The network is defined in `main.netrun.json`:

In [ ]:
print(json.dumps(config_data, indent=2))